:meth:`SequenceFeature.prune_by_correlation` removes **empirically redundant** features: among features whose realized values are pairwise correlated beyond ``max_cor``, it keeps the one with the higher ``abs_auc`` and drops the others. The correlation is measured over the actual samples, which makes it complementary to CPP's in-run redundancy reduction (that one compares the underlying scale vectors). Use it after :meth:`SequenceFeature.prune_by_variance` and before :meth:`TreeModel.select_features`.

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=20)
labels = df_seq["label"].to_list()
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
df_feat = aa.CPP(df_parts=df_parts).run(labels=labels, n_filter=50)
print(f"features from CPP.run: {len(df_feat)}")

features from CPP.run: 50


At ``max_cor=0.7`` no retained pair of features has an absolute correlation above 0.7. The output is sorted by descending ``abs_auc`` and is deterministic across runs:

In [2]:
df_cor = sf.prune_by_correlation(df_feat=df_feat, df_parts=df_parts, max_cor=0.7)
print(f"kept {len(df_cor)} of {len(df_feat)} features")
df_cor.head()

kept 21 of 50 features


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
0,"TMD-Pattern(C,4,8)-BEGF750101",Conformation,α-helix,α-helix,Conformational parameter of inner helix (Beghi...,0.444,0.299,0.299,0.090,0.196,0.000002,0.022853,"23,27"
1,"TMD_C_JMD_C-PeriodicPattern(C,i+4/4,2)-BEGF750103",Conformation,β-turn,β-turn,Conformational parameter of beta-turn (Beghin-...,0.409,0.139,-0.139,0.098,0.094,0.000010,0.011310,"23,27,31,35,39"
2,"TMD_C_JMD_C-Pattern(C,8,11,14)-ZIMJ680104",Energy,Isoelectric point,Isoelectric point,"Isoelectric point (Zimmerman et al., 1968)",0.402,0.130,0.130,0.080,0.088,0.000013,0.011349,"27,30,33"
3,"TMD_C_JMD_C-Pattern(C,10,14)-MUNV940102",Energy,Free energy (folding),Free energy (α-helix),Free energy in alpha-helical region (Munoz-Ser...,0.391,0.153,-0.153,0.067,0.130,0.000023,0.012601,"27,31"
4,"TMD-Segment(10,11)-CHAM820102",Polarity,Hydrophobicity (interface),Free energy (interface),"Free energy of solution in water, kcal/mole (C...",0.389,0.169,-0.169,0.050,0.130,0.000026,0.012145,"27,28"


Lower ``max_cor`` to prune more aggressively. As with variance pruning, a pre-computed ``X`` can be supplied (``df_parts`` / ``df_scales`` then ignored), and ``accept_gaps`` / ``n_jobs`` are forwarded to the matrix build:

In [3]:
X = sf.feature_matrix(features=df_feat, df_parts=df_parts, accept_gaps=False, n_jobs=1)
df_cor_tight = sf.prune_by_correlation(df_feat=df_feat, X=X, max_cor=0.4)
print(f"max_cor=0.4 kept {len(df_cor_tight)} features; max_cor=0.7 kept {len(df_cor)}")

max_cor=0.4 kept 2 features; max_cor=0.7 kept 21


The two pruners compose, and the result drops straight into model-based selection (:meth:`TreeModel.select_features`):

In [4]:
df_pruned = sf.prune_by_variance(df_feat=df_feat, df_parts=df_parts, threshold=0.0)
df_pruned = sf.prune_by_correlation(df_feat=df_pruned, df_parts=df_parts, max_cor=0.5)
print(f"variance -> correlation kept {len(df_pruned)} of {len(df_feat)} features")
df_pruned.head()

variance -> correlation kept 5 of 50 features


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
0,"TMD-Pattern(C,4,8)-BEGF750101",Conformation,α-helix,α-helix,Conformational parameter of inner helix (Beghi...,0.444,0.299,0.299,0.090,0.196,0.000002,0.022853,"23,27"
1,"TMD_C_JMD_C-Pattern(C,8,11,14)-ZIMJ680104",Energy,Isoelectric point,Isoelectric point,"Isoelectric point (Zimmerman et al., 1968)",0.402,0.130,0.130,0.080,0.088,0.000013,0.011349,"27,30,33"
2,"TMD_C_JMD_C-PeriodicPattern(N,i+4/3,1)-AURR980108",Conformation,α-helix,"α-helix (N-terminal, inside)",Normalized positional residue frequency at hel...,0.385,0.156,0.156,0.077,0.105,0.000031,0.012681,"21,25,28,32,35,39"
3,"JMD_N_TMD_N-Segment(3,8)-NAKH920107",Composition,AA composition,"Membrane proteins (multi-spanning, EXT)",AA composition of EXT of multi-spanning protei...,0.366,0.188,0.188,0.144,0.094,0.000074,0.009765,"6,7"
4,"TMD_C_JMD_C-Pattern(C,8,12,15)-GEIM800103",Conformation,Unclassified (Conformation),α-helix (β-proteins),Alpha-helix indices for beta-proteins (Geisow-...,0.359,0.163,-0.163,0.137,0.080,0.000104,0.009706,"26,29,33"


**What can go wrong?** A ``df_feat`` with fewer than two features has nothing to compare and is returned unchanged:

In [5]:
one = df_feat.head(1)
print(f"single-feature input -> {len(sf.prune_by_correlation(df_feat=one, df_parts=df_parts))} feature")

single-feature input -> 1 feature
